Ok so from notebook 4 we saw that VADER does miss some key words and so it misses sentiment, but it is not just that -VADER also misses HIGH SEVERITY FINTECH PROBLEMS, and those misses differ by app


From the VADER baseline in Notebook 4, we learned that VADER does not just miss sentiment — it misses high‑severity fintech failures, and those misses differ significantly by app.

This means that a simple lexicon model can classify a review as “Neutral” even when the user is describing a severe operational failure such as account lockouts, frozen funds, verification loops, transfer delays, or card declines.

These are the exact types of failures that cost fintech companies money, trust, and retention — and they are hidden inside VADER’s 26% mismatch rate.

In this notebook, we build a Fintech Severity Engine to quantify the severity of each review and identify which apps have the most hidden high‑severity complaints

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

pd.set_option('display.max_colwidth', 120)
sns.set_style('whitegrid')

print("libraries loaded successfully")

In [ ]:
df = pd.read_csv("../data/clean/all_apps_clean.csv")

analyzer = SentimentIntensityAnalyzer()

def get_vader_score(text):
    if pd.isna(text):
        return 0.0
    return analyzer.polarity_scores(str(text))['compound']

def vader_label(compound):
    if compound >= 0.05:
        return 'Positive'
    elif compound <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

def rating_label(rating):
    if rating >= 4:
        return 'Positive'
    elif rating <= 2:
        return 'Negative'
    else:
        return 'Neutral'

df['vader_compound'] = df['review_clean'].apply(get_vader_score)
df['vader_sentiment'] = df['vader_compound'].apply(vader_label)
df['rating_sentiment'] = df['rating'].apply(rating_label)

print(f"Dataset loaded: {df.shape}")
display(df[['app_name', 'rating', 'review_clean', 'vader_compound', 'vader_sentiment', 'rating_sentiment']].head())

NOW LET US BUILD THE SEVERITY ENGINE



In [ ]:
severity_keywords = {
    5: ['locked out', 'account locked', 'account closed', 'money missing',
        'funds missing', 'stolen', 'fraud', 'scam', 'cannot access',
        'lost my money', 'money gone', 'unauthorized'],
    4: ['declined', 'card declined', 'transfer failed', 'verification failed',
        'failed', 'frozen', 'pending', 'verification', "can't send",
        "can't receive", 'not working', 'blocked'],
    3: ['customer service', 'support', 'refund', 'delay', 'late',
        'problem', 'issue', 'error', 'wrong', 'charge'],
    2: ['slow', 'bug', 'glitch', 'annoying', 'confusing', 'difficult'],
    1: ['minor', 'small', 'slight', 'okay', 'fine']
}

def compute_severity(text):
    if pd.isna(text): return 1
    text = str(text).lower()
    for level in [5, 4, 3, 2, 1]:
        for kw in severity_keywords[level]:
            if kw in text:
                return level
    return 1

severity_map = {1:'Low', 2:'Low-Moderate', 3:'Moderate', 4:'High', 5:'Critical'}

df['severity_score'] = df['review_clean'].apply(compute_severity)
df['severity_label'] = df['severity_score'].map(severity_map)

print("Severity distribution (all reviews):")
print(df['severity_label'].value_counts())

Create hidden_neg AFTER severity exist

In [ ]:
# Must run AFTER Cell 2 so severity columns exist in df
hidden_neg = df[
    (df['rating_sentiment'] == 'Negative') &
    (df['vader_sentiment']  != 'Negative')
].copy()

total_true_neg   = (df['rating_sentiment'] == 'Negative').sum()
hidden_neg_count = len(hidden_neg)
hidden_neg_rate  = hidden_neg_count / total_true_neg

print(f"Total true-negative reviews    : {total_true_neg:,}")
print(f"Hidden negatives (VADER missed): {hidden_neg_count:,}")
print(f"Hidden negative rate           : {hidden_neg_rate:.1%}")
print(f"\nBreakdown of what VADER called them instead:")
print(hidden_neg['vader_sentiment'].value_counts())
print(f"\nSeverity breakdown inside hidden negatives:")
print(hidden_neg['severity_label'].value_counts())

NOW LET US BUILD THE SEVERITY ENGINE

Now let us APPLY SEVERITY TO HIDDEN NEGATIVES + VISUALIZE

In [ ]:
sev_counts = hidden_neg['severity_label'].value_counts().reindex(
    ['Critical','High','Moderate','Low-Moderate','Low'], fill_value=0
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sev_counts.plot(kind='bar', ax=axes[0], color='#c0392b', edgecolor='black')
axes[0].set_title("Severity of Hidden Negative Reviews\n(Reviews VADER Missed)", fontsize=13)
axes[0].set_xlabel("Severity")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=30)

sev_miss_rate = (
    df[df['rating_sentiment'] == 'Negative']
    .groupby('severity_label')
    .apply(lambda x: (x['vader_sentiment'] != 'Negative').mean())
    .reindex(['Critical','High','Moderate','Low-Moderate','Low'])
)

sev_miss_rate.plot(kind='bar', ax=axes[1], color='#e67e22', edgecolor='black')
axes[1].set_title("VADER Miss Rate by Severity\n(% of True Negatives Missed)", fontsize=13)
axes[1].set_xlabel("Severity")
axes[1].set_ylabel("Miss Rate")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig("severity_hidden_negatives.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nCritical hidden negatives : {sev_counts.get('Critical', 0):,}")
print(f"High hidden negatives     : {sev_counts.get('High', 0):,}")